# AoC 2024 Day 6 — Guard Gallivant

**Python — sequential guard walk**

Puzzle: <https://adventofcode.com/2024/day/6>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A map of the lab. `#` marks an obstruction, `^` is a guard standing on an open cell and facing up, and `.` is open floor.

The guard follows one rule on repeat: if the cell directly ahead is blocked, turn right 90°; otherwise step into it. Sooner or later she walks off the edge of the map.

- **Part 1** — count the distinct cells she stands on before leaving, including the one she started from.

## The approach

This one is solved in **plain Python inside a Spark project**, and that is a deliberate call rather than a shortcut.

The guard's rule set is two lines long, but it is a *state machine*: her position and facing at step *k* are a function of her position and facing at step *k−1*. There is no row of a "guard positions" table that can be computed without first computing the row above it, and no key to partition on — the whole puzzle is one chain 6082 steps long, with 158 turns in it.

What would a Spark version actually cost? The best relational framing is the one `reference_python/y2024/day06.py` uses: stop stepping cell by cell and *jump to the next obstacle*, so the patrol becomes 158 legs instead of 6082 steps. Each leg is a query — "the nearest `#` in this column above this row" — over a 130×130 = 16,900 row cell relation, and the answer has to come back to the driver before the next leg can even be described. That is 158 round trips through Spark Connect. At the tens of milliseconds a trivial Spark job costs, that is several seconds of pure latency; the Python loop finishes the entire patrol in **2.2 ms**.

Iterating a frontier in Spark pays off when each round does a *lot* of work in parallel — day 10 in this repo is exactly that shape. Here each round decides the location of a single guard. There is nothing to parallelise, so the distributed framing buys latency and buys back nothing.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day06

spark = get_spark('aoc-2024-day06')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '....#.....\n.........#\n..........\n..#.......\n.......#..\n..........\n.#..^.....\n........#.\n#.........\n......#...\n'

print('part 1:', day06.part1(spark, EXAMPLE), '(expected 41)')

### The state you cannot skip ahead in

The cell below replays the patrol and prints the first few turns in order. Read them as a dependency chain: the map alone does not tell you where turn 4 happens — turns 1 through 3 do.

In [ ]:
from aoc_spark.y2024.day06 import TURNS

grid = EXAMPLE.strip().splitlines()
height, width = len(grid), len(grid[0])
obstacles = {(r, c) for r, line in enumerate(grid) for c, ch in enumerate(line) if ch == '#'}
start = next((r, c) for r, line in enumerate(grid) for c, ch in enumerate(line) if ch == '^')
print(f'{height}x{width} grid, {len(obstacles)} obstacles, start at {start}')

# Replay the walk, recording every turn. Each entry below is only reachable
# because the entry above it already happened.
r, c = start
d = 0
seen = {start}
turns = []
while True:
    dr, dc = TURNS[d]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < height and 0 <= nc < width):
        break
    if (nr, nc) in obstacles:
        turns.append((len(seen), (r, c), TURNS[d], TURNS[(d + 1) % 4]))
        d = (d + 1) % 4
        continue
    r, c = nr, nc
    seen.add((r, c))

print(f'{len(turns)} turns, {len(seen)} distinct positions')
for n_seen, at, facing, now in turns[:6]:
    print(f'  after {n_seen:2d} cells seen: blocked at {at}, {facing} -> {now}')

# The path, overlaid on the map -- compare with the puzzle's "X" diagram.
print()
for rr in range(height):
    print(''.join('#' if (rr, cc) in obstacles else 'X' if (rr, cc) in seen else '.'
                  for cc in range(width)))

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 6)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day06.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day06 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- `TURNS` is ordered up, right, down, left, which is the *only* reason `(d + 1) % 4` means "turn right". Reorder that list and the guard quietly starts turning left.
- The turn branch `continue`s **without moving**. A guard boxed in on two sides turns twice on the spot; folding turn-and-step into one iteration breaks exactly that case.
- `seen` is seeded with `start` before the loop, because the puzzle counts the starting cell. Seed it empty and every answer is one low.
- Assumes a rectangular grid — `width` is taken from row 0 and applied to all rows — and exactly one `^`. `next(...)` raises `StopIteration` if the start marker is missing.
- Assumes the guard **leaves**. There is no step budget, so a map that traps her in a cycle hangs the notebook rather than erroring. That is safe for part 1 by construction; it is not a property to rely on if you start editing the map.
- `data.strip()` removes the trailing newline. It would also eat leading blank lines, which would shift every row index — the real input has none.